# Pipeline: Intervention Plan Success Prediction

## Problem framing
**Who cares**: Case management lead / social workers.

**Business question**: Which intervention plans are likely to succeed ("Achieved") vs stall ("On Hold" / stay "Open"), so staff can adjust plans early.

**Predictive goal**: Classify whether an intervention plan will reach "Achieved" status based on the resident's profile, plan characteristics, services provided, and concurrent activity (sessions, visits, incidents).

## Data used
- `intervention_plans.csv` — 180 plans with status, category, services, target values
- `residents.csv` — full resident profile including subcategory flags, family context, risk levels
- `process_recordings.csv` — session counts/types per resident
- `home_visitations.csv` — visit counts per resident
- `incident_reports.csv` — incident counts per resident

## Output
- `predictions_intervention_success.csv` — per-plan predictions with resident context

In [1]:
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 200)

HERE = Path.cwd()
if (HERE / "intervention_plans.csv").exists():
    DATA_DIR = HERE
elif (HERE.parent / "intervention_plans.csv").exists():
    DATA_DIR = HERE.parent
else:
    raise FileNotFoundError("Could not locate intervention_plans.csv")

plans = pd.read_csv(DATA_DIR / "intervention_plans.csv")
residents = pd.read_csv(DATA_DIR / "residents.csv")
incidents = pd.read_csv(DATA_DIR / "incident_reports.csv")
visits = pd.read_csv(DATA_DIR / "home_visitations.csv")
sessions = pd.read_csv(DATA_DIR / "process_recordings.csv")
safehouses = pd.read_csv(DATA_DIR / "safehouses.csv")

plans["created_at"] = pd.to_datetime(plans["created_at"], errors="coerce")
plans["target_date"] = pd.to_datetime(plans["target_date"], errors="coerce")
incidents["incident_date"] = pd.to_datetime(incidents["incident_date"], errors="coerce")
visits["visit_date"] = pd.to_datetime(visits["visit_date"], errors="coerce")
sessions["session_date"] = pd.to_datetime(sessions["session_date"], errors="coerce")

print(f"Plans: {len(plans)}, Residents: {len(residents)}")
print(f"Plan status distribution:\n{plans['status'].value_counts().to_string()}")

Plans: 180, Residents: 60
Plan status distribution:
status
In Progress    72
Open           39
On Hold        37
Achieved       29
Closed          3


In [2]:
# Build features per intervention plan

# Label: 1 = Achieved, 0 = not achieved (In Progress / Open / On Hold / Closed)
plans["achieved"] = (plans["status"] == "Achieved").astype(int)

# Plan-level features
plans["days_to_target"] = (plans["target_date"] - plans["created_at"]).dt.days
plans["target_value"] = pd.to_numeric(plans["target_value"], errors="coerce")

# Parse services_provided into count + individual flags
plans["n_services"] = plans["services_provided"].fillna("").str.split(",").apply(len)
all_services = set()
for s in plans["services_provided"].dropna():
    for svc in s.split(","):
        all_services.add(svc.strip())

for svc in sorted(all_services):
    col = f"svc_{svc.lower().replace(' ', '_')}"
    plans[col] = plans["services_provided"].fillna("").str.contains(svc, case=False).astype(int)

# Resident-level activity aggregates (total before plan creation)
def resident_aggs(plan_row):
    rid = plan_row["resident_id"]
    before = plan_row["created_at"]
    
    inc = incidents[(incidents["resident_id"] == rid) & (incidents["incident_date"] < before)]
    vis = visits[(visits["resident_id"] == rid) & (visits["visit_date"] < before)]
    sess = sessions[(sessions["resident_id"] == rid) & (sessions["session_date"] < before)]
    
    return pd.Series({
        "incidents_before": len(inc),
        "high_sev_before": (inc["severity"].astype(str).str.lower() == "high").sum() if len(inc) > 0 else 0,
        "visits_before": len(vis),
        "sessions_before": len(sess),
        "session_minutes_before": sess["session_duration_minutes"].fillna(0).sum() if len(sess) > 0 else 0,
        "concerns_before": sess["concerns_flagged"].fillna(False).astype(bool).sum() if len(sess) > 0 else 0,
    })

activity = plans.apply(resident_aggs, axis=1)
df = pd.concat([plans, activity], axis=1)

# Resident static features (full profile including subcategories)
res_cols = [
    "resident_id", "safehouse_id", "case_status", "case_category",
    "initial_risk_level", "current_risk_level", "referral_source",
    "has_special_needs", "is_pwd", "sex",
    "sub_cat_orphaned", "sub_cat_trafficked", "sub_cat_child_labor",
    "sub_cat_physical_abuse", "sub_cat_sexual_abuse", "sub_cat_osaec",
    "sub_cat_cicl", "sub_cat_at_risk", "sub_cat_street_child",
    "family_is_4ps", "family_solo_parent", "family_indigenous",
    "family_informal_settler", "age_upon_admission", "present_age",
]
res_cols = [c for c in res_cols if c in residents.columns]
df = df.merge(residents[res_cols], on="resident_id", how="left")

print(f"Model dataset: {len(df)} rows, {df.shape[1]} columns")
print(f"Label balance: {df['achieved'].mean():.1%} achieved")
df.head()

Model dataset: 180 rows, 48 columns
Label balance: 16.1% achieved


,plan_id,resident_id,plan_category,plan_description,services_provided,target_value,target_date,status,case_conference_date,created_at,updated_at,achieved,days_to_target,n_services,svc_caring,svc_healing,svc_legal_services,svc_teaching,incidents_before,high_sev_before,visits_before,sessions_before,session_minutes_before,concerns_before,safehouse_id,case_status,case_category,initial_risk_level,current_risk_level,referral_source,has_special_needs,is_pwd,sex,sub_cat_orphaned,sub_cat_trafficked,sub_cat_child_labor,sub_cat_physical_abuse,sub_cat_sexual_abuse,sub_cat_osaec,sub_cat_cicl,sub_cat_at_risk,sub_cat_street_child,family_is_4ps,family_solo_parent,family_indigenous,family_informal_settler,age_upon_admission,present_age
0,1,1,Safety,Maintain a stable and safe environment,"Healing, Legal Services, Teaching",4.20,2024-02-01,On Hold,2023-11-01,2023-10-01,2024-03-01 00:00:00,0,123,3,0,1,1,1,0,0,0,0,0,0,4,Active,Neglected,Critical,High,NGO,True,False,F,False,False,False,False,False,False,False,False,False,False,False,False,False,15 Years 9 months,17 Years 6 months
1,2,1,Education,Improve participation and course completion,"Caring, Legal Services, Healing",0.85,2024-02-01,In Progress,2024-01-30,2023-10-01,2024-03-01 00:00:00,0,123,3,1,1,1,0,0,0,0,0,0,0,4,Active,Neglected,Critical,High,NGO,True,False,F,False,False,False,False,False,False,False,False,False,False,False,False,False,15 Years 9 months,17 Years 6 months
2,3,1,Physical Health,Improve nutrition and overall wellbeing,"Teaching, Healing, Caring",4.20,2024-02-01,On Hold,2023-10-24,2023-10-01,2024-03-01 00:00:00,0,123,3,1,1,0,1,0,0,0,0,0,0,4,Active,Neglected,Critical,High,NGO,True,False,F,False,False,False,False,False,False,False,False,False,False,False,False,False,15 Years 9 months,17 Years 6 months
3,4,2,Safety,Maintain a stable and safe environment,Legal Services,4.20,2023-11-01,On Hold,2023-11-02,2023-03-01,2023-12-01 00:00:00,0,245,1,0,0,1,0,0,0,0,0,0,0,3,Closed,Surrendered,Medium,Medium,Government Agency,False,False,F,False,False,False,False,False,False,False,True,True,False,False,True,False,15 Years 5 months,17 Years 10 months
4,5,2,Education,Improve participation and course completion,"Healing, Caring",0.85,2023-11-01,Open,2023-04-26,2023-03-01,2023-12-01 00:00:00,0,245,2,1,1,0,0,0,0,0,0,0,0,3,Closed,Surrendered,Medium,Medium,Government Agency,False,False,F,False,False,False,False,False,False,False,True,True,False,False,True,False,15 Years 5 months,17 Years 10 months


In [3]:
# Model training

TARGET = "achieved"
exclude = {"plan_id", "resident_id", "plan_description", "services_provided",
           "case_conference_date", "created_at", "updated_at", "target_date",
           "status", TARGET, "safehouse_id"}
feature_cols = [c for c in df.columns if c not in exclude]

X_all = df[feature_cols].copy()
y_all = df[TARGET].astype(int)

can_stratify = y_all.nunique() > 1 and y_all.value_counts().min() >= 2
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42,
    stratify=y_all if can_stratify else None,
)

numeric_cols = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in numeric_cols]

for c in cat_cols:
    X_train[c] = X_train[c].astype("object")
    X_test[c] = X_test[c].astype("object")

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
])

models = {
    "LogisticRegression": LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=300, max_depth=10, class_weight="balanced", random_state=42, n_jobs=-1),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
best_name, best_score, best_pipe = None, -1e9, None

for name, model in models.items():
    p = Pipeline([("pre", pre), ("model", model)])
    scores = cross_val_score(p, X_train, y_train, cv=cv, scoring="roc_auc")
    print(f"{name}: CV ROC-AUC = {scores.mean():.3f} ± {scores.std():.3f}")
    if scores.mean() > best_score:
        best_name, best_score, best_pipe = name, scores.mean(), p

print(f"\nBest model: {best_name}")
best_pipe.fit(X_train, y_train)
pipe = best_pipe

proba = pipe.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

if y_test.nunique() >= 2:
    print(f"Test ROC-AUC: {roc_auc_score(y_test, proba):.3f}")
    print(f"Test PR-AUC:  {average_precision_score(y_test, proba):.3f}")
print(classification_report(y_test, pred, zero_division=0))

LogisticRegression: CV ROC-AUC = 0.654 ± 0.073


RandomForest: CV ROC-AUC = 0.605 ± 0.045

Best model: LogisticRegression
Test ROC-AUC: 0.661
Test PR-AUC:  0.265
              precision    recall  f1-score   support

           0       0.87      0.87      0.87        30
           1       0.33      0.33      0.33         6

    accuracy                           0.78        36
   macro avg       0.60      0.60      0.60        36
weighted avg       0.78      0.78      0.78        36



In [4]:
# Feature importance

ohe = pipe.named_steps["pre"].named_transformers_["cat"].named_steps["ohe"]
cat_feature_names = ohe.get_feature_names_out(cat_cols).tolist() if len(cat_cols) else []
feat_names = numeric_cols + cat_feature_names

model = pipe.named_steps["model"]
if hasattr(model, "coef_"):
    imp = model.coef_[0]
    label = "coef"
else:
    imp = model.feature_importances_
    label = "importance"

imp_df = pd.DataFrame({"feature": feat_names, label: imp}).sort_values(label, key=abs, ascending=False)
print(f"Top 20 features by |{label}| ({best_name})")
display(imp_df.head(20))

Top 20 features by |coef| (LogisticRegression)


,feature,coef
43,current_risk_level_High,1.226719
47,referral_source_Court Order,1.159027
96,present_age_13 Years 7 months,1.154204
55,age_upon_admission_12 Years 2 months,1.087822
13,has_special_needs,-0.985338
57,age_upon_admission_12 Years 4 months,0.939212
95,present_age_13 Years 2 months,0.939212
46,referral_source_Community,-0.838291
36,case_category_Neglected,-0.771324
41,initial_risk_level_Medium,0.731783


In [5]:
# Generate predictions for all plans (especially In Progress / Open / On Hold)

X_all_pred = df[feature_cols].copy()
for c in cat_cols:
    if c in X_all_pred.columns:
        X_all_pred[c] = X_all_pred[c].astype("object")

df["pred_success_probability"] = pipe.predict_proba(X_all_pred)[:, 1]

p75 = df["pred_success_probability"].quantile(0.75)
p35 = df["pred_success_probability"].quantile(0.35)

def tier(p):
    if p >= p75: return "Likely to Succeed"
    elif p >= p35: return "Moderate"
    else: return "At Risk"

df["prediction_tier"] = df["pred_success_probability"].apply(tier)

# Join safehouse names
df = df.merge(safehouses[["safehouse_id", "safehouse_code", "name"]], on="safehouse_id", how="left")

# Focus output on plans that are still active
active = df[df["status"].isin(["In Progress", "Open", "On Hold"])].copy()

out_cols = [
    "plan_id", "resident_id", "safehouse_code", "name",
    "plan_category", "status", "prediction_tier", "pred_success_probability",
    "services_provided", "n_services", "target_value", "days_to_target",
    "sessions_before", "visits_before", "incidents_before",
    "current_risk_level", "case_category",
]
out_cols = [c for c in out_cols if c in active.columns]

out = active[out_cols].sort_values("pred_success_probability").copy()

out_path = DATA_DIR / "predictions_intervention_success.csv"
out.to_csv(out_path, index=False)

print(f"Saved: {out_path}")
print(f"Active plans: {len(out)} ({len(active[active['prediction_tier']=='At Risk'])} at risk)")
print(f"\nTier distribution (active plans):")
print(out["prediction_tier"].value_counts().to_string())
display(out.head(15))

Saved: /Users/masonzarges/Desktop/INTEX Data/predictions_intervention_success.csv
Active plans: 148 (60 at risk)

Tier distribution (active plans):
prediction_tier
Moderate             69
At Risk              60
Likely to Succeed    19


,plan_id,resident_id,safehouse_code,name,plan_category,status,prediction_tier,pred_success_probability,services_provided,n_services,target_value,days_to_target,sessions_before,visits_before,incidents_before,current_risk_level,case_category
153,154,52,SH02,Lighthouse Safehouse 2,Safety,In Progress,At Risk,0.000860,"Caring, Legal Services, Teaching",3,4.20,151,0,0,0,Low,Abandoned
154,155,52,SH02,Lighthouse Safehouse 2,Education,In Progress,At Risk,0.001428,"Legal Services, Caring, Teaching",3,0.85,151,0,0,0,Low,Abandoned
126,127,43,SH02,Lighthouse Safehouse 2,Safety,On Hold,At Risk,0.002243,"Legal Services, Teaching, Caring",3,4.20,153,0,0,0,Low,Neglected
127,128,43,SH02,Lighthouse Safehouse 2,Education,In Progress,At Risk,0.005539,"Caring, Healing, Teaching",3,0.85,153,0,0,0,Low,Neglected
57,58,20,SH01,Lighthouse Safehouse 1,Safety,Open,At Risk,0.005859,"Caring, Teaching, Legal Services",3,4.20,214,0,0,0,Medium,Neglected
75,76,26,SH03,Lighthouse Safehouse 3,Safety,In Progress,At Risk,0.005900,"Legal Services, Teaching, Caring",3,4.20,184,0,0,0,Low,Abandoned
25,26,9,SH07,Lighthouse Safehouse 7,Education,In Progress,At Risk,0.007024,Teaching,1,0.85,214,0,0,0,Low,Surrendered
156,157,53,SH06,Lighthouse Safehouse 6,Safety,Open,At Risk,0.007511,Teaching,1,4.20,305,0,0,0,Low,Surrendered
147,148,50,SH04,Lighthouse Safehouse 4,Safety,In Progress,At Risk,0.007595,"Teaching, Caring",2,4.20,304,0,0,0,Medium,Neglected
24,25,9,SH07,Lighthouse Safehouse 7,Safety,Open,At Risk,0.007776,"Caring, Healing, Legal Services",3,4.20,214,0,0,0,Low,Surrendered
